# Programación Concurrente en Python — Clase 3
## Semáforos, Deadlocks y Sincronización Avanzada

Clase anterior:
- Condiciones de carrera
- Cómo usar locks para proteger secciones críticas
- Por qué los hilos pueden producir resultados incorrectos sin sincronización

Hoy:
- Semáforos
- Deadlocks (y cómo evitarlos)
- Sincronización de múltiples hilos

Trabajaremos en situaciones donde los locks simples no son suficientes o pueden producir bloqueos indeseados.

In [1]:
#@title Descripción de race conditions, locks y deadlocks
%%html
<iframe width="560" height="315" src="https://www.youtube.com/embed/BEyzoRtQDEs?si=OAUHbsATebbAjtsJ" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>

SyntaxError: invalid syntax (3444286290.py, line 3)

In [ ]:
#@title Definición de los deadlocks
%%html
<iframe width="560" height="315" src="https://www.youtube.com/embed/y7DOHyBTWps?si=aRuMO4v3nr0Gyzn7" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>

## Recordatorio de clase pasada:

### Un lock:
- Garantiza que solo un hilo entre a una sección crítica.
- Es ideal para evitar condiciones de carrera simples.

El prolema es que también puede haber problemas si se usan múltiples locks de forma descontrolada.

## Métodos a usar de la librería *threading*

1. threading.Semaphore()

   Un semáforo permite que una cantidad limitada de hilos entren a una región crítica.
   A diferencia de un lock (que permite solo 1 hilo), el semáforo puede permitir N hilos.

   Ejemplo:
       sem = threading.Semaphore(3)
   Esto permite que hasta 3 hilos trabajen simultáneamente en una sección.

2. threading.RLock()

   Un RLock (Reentrant Lock) es un tipo especial de lock que puede ser adquirido
   varias veces por el MISMO hilo sin bloquearlo.
   Es útil cuando una función protegida llama a otra función que también usa el mismo lock.

3. Deadlocks

   Ocurren cuando dos o más hilos se quedan esperando mutuamente locks que nunca se liberan.

# SEMÁFOROS

Un semáforo es similar a un lock, pero con un contador interno. En lugar de permitir 1 hilo, permite N hilos simultáneamente.

## Ejemplo cotidiano:
 - Un estacionamiento con 5 espacios disponibles: Solo 5 coches pueden entrar al mismo tiempo.
 - Un semáforo representaría estos 5 espacios.



In [3]:
# Importemos las librerías
import threading
import time

In [4]:
# Creemos un semáforo que permita 3 hilos simultáneamente

semaforo = threading.Semaphore(3)

def tarea(id):
    with semaforo:
        print(f"Hilo {id} Entrando\n")
        time.sleep(1)
        print(f"Hilo {id} Saliendo\n")

hilos = []
for i in range(8):
    t = threading.Thread(target=tarea, args=(i,))
    t.start()
    hilos.append(t)

for t in hilos:
    t.join()
print("Semáforo completado.")
print("Observemos que no hay un orden en las entradas o salidas. El acomodo es totalmente estocástico.")


Hilo 0 Entrando

Hilo 1 Entrando

Hilo 2 Entrando

Hilo 0 Saliendo

Hilo 1 Saliendo

Hilo 4 Entrando

Hilo 3 Entrando

Hilo 2 Saliendo

Hilo 5 Entrando

Hilo 4 Saliendo
Hilo 3 Saliendo

Hilo 5 Saliendo

Hilo 6 Entrando


Hilo 7 Entrando

Hilo 6 Saliendo

Hilo 7 Saliendo

Semáforo completado.
Observemos que no hay un orden en las entradas o salidas. El acomodo es totalmente estocástico.


#### A veces una función protegida por un lock llama otra función que también usa el mismo lock.Con un lock normal, esto produce un bloqueo (deadlock), porque el hilo intenta adquirir el mismo lock dos veces.

#### En esos casos debemos usar RLock (reentrant lock), que permite que el MISMO hilo adquiera el candado varias veces.

```
lock_normal = threading.Lock()
lock_r = threading.RLock()
```



## Lock normal causando problemas

Veamos un ejemplo que falla con Lock pero funciona con RLock.

In [ ]:
lock = threading.Lock()

def funcion_b():
    # Intentamos adquirir el lock de nuevo
    with lock:
        print("funcion_b ejecutándose")

def funcion_a():
    with lock:
        print("funcion_a ejecutándose")
        # funcion_a llama a funcion_b que intenta volver a adquirir el lock
        funcion_b()

print("Ejecutando ejemplo con Lock (esto se bloqueará)...")

t = threading.Thread(target=funcion_a)
t.start()
t.join(timeout=10) # Con este timeout forzamos a que el programa se cierre y no se quede pensando
print("No se imprimió la ejecución de la función b")
print("Significa que el programa no se completa porque se produjo un deadlock interno.\n")

Ejecutando ejemplo con Lock (esto se bloqueará)...
funcion_a ejecutándose
No se imprimió la ejecución de la función b
Significa que el programa no se completa porque se produjo un deadlock interno.



# RLock solucionando el problema

In [ ]:
# Repitamos el ejemplo anterior usando RLock para ver la diferencia.

#import threading
#import time

rlock = threading.RLock() #OJO: Aquí está la diferencia

def funcion_b():
    with rlock:
        print("funcion_b ejecutándose")

def funcion_a():
    with rlock:
        print("funcion_a ejecutándose")
        funcion_b()

print("Ejecutando ejemplo con RLock (funciona correctamente)...")

t = threading.Thread(target=funcion_a)
t.start()
t.join()

print("Ejemplo completado sin deadlock.")

Ejecutando ejemplo con RLock (funciona correctamente)...
funcion_a ejecutándose
funcion_b ejecutándose
Ejemplo completado sin deadlock.


#DEADLOCKS
Un deadlock ocurre cuando dos o más hilos quedan esperando recursos que nunca se liberarán.

### Ejemplo clásico:
 - Hilo 1 adquiere lock A y espera lock B.
 - Hilo 2 adquiere lock B y espera lock A.

Por lo tanto, ninguno avanza.

In [ ]:
#import threading
#import time

lock_a = threading.Lock()
lock_b = threading.Lock()

def hilo1():
    with lock_a:
        print("Hilo 1 adquirió A")
        time.sleep(0.2)
        print("Hilo 1 intentando adquirir B...")
        with lock_b:
            print("Hilo 1 adquirió B")

def hilo2():
    with lock_b:
        print("Hilo 2 adquirió B")
        time.sleep(0.2)
        print("Hilo 2 intentando adquirir A...")
        with lock_a:
            print("Hilo 2 adquirió A")

print("Este programa puede quedar bloqueado (deadlock).")

t1 = threading.Thread(target=hilo1)
t2 = threading.Thread(target=hilo2)

t1.start()
t2.start()

t1.join(timeout=2)
t2.join(timeout=2)

print("Si no se imprimieron ambas adquisiciones, ocurrió un deadlock.")

Este programa puede quedar bloqueado (deadlock).
Hilo 1 adquirió A
Hilo 2 adquirió B
Hilo 1 intentando adquirir B...
Hilo 2 intentando adquirir A...
Si no se imprimieron ambas adquisiciones, ocurrió un deadlock.


# Cómo evitar deadlocks?
 1. Adquirir los locks SIEMPRE en el mismo orden. Todos los hilos: primero A, luego B, nunca al revés.

 2. Usar timeouts para evitar esperar indefinidamente.

 3. Minimizar el número de locks anidados.

 4. Usar RLocks cuando sea seguro.

 5. Analizar cuidadosamente qué secciones realmente necesitan un lock.

# Corrección de deadlock

In [ ]:
import threading
import time

lock_a = threading.Lock()
lock_b = threading.Lock()

def hilo1():
    # Siempre adquirimos lock A y luego lock B
    with lock_a:
        print("Hilo 1 adquirió A")
        time.sleep(0.2)
        with lock_b:
            print("Hilo 1 adquirió B")

def hilo2():
    # También adquirimos A primero y luego B
    with lock_a:
        print("Hilo 2 adquirió A")
        time.sleep(0.2)
        with lock_b:
            print("Hilo 2 adquirió B")

print("Ejemplo sin deadlock:")

t1 = threading.Thread(target=hilo1)
t2 = threading.Thread(target=hilo2)

t1.start()
t2.start()

t1.join()
t2.join()

print("Todas las adquisiciones se completaron correctamente.")

Ejemplo sin deadlock:
Hilo 1 adquirió A
Hilo 1 adquirió B
Hilo 2 adquirió A
Hilo 2 adquirió B
Todas las adquisiciones se completaron correctamente.


# Conclusión:
 - Los semáforos permiten controlar cuántos hilos pueden entrar a una sección.
 - Los locks simples pueden causar deadlocks si no se usan correctamente.
 - Un RLock permite readquirir el candado dentro del mismo hilo.
 - Los deadlocks pueden evitarse con orden consistente, timeouts y buen diseño.

# Ejercicio de Práctica:
Simular un sistema de impresión con varios documentos, en el que **solo 2 impresoras están disponibles**. 10 hilos deben representar a usuarios intentando imprimir.
### TAREA:
- Implementar la versión con un Semaphore que permita 2 impresiones simultáneas.
- Añadir sleeps pequeños para simular impresión.
- En una sección aparte, u otro archivo .py, modificar el código para mostrar qué ocurre si no se usa semáforo.